In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import random as rd

In [3]:
train=pd.read_csv("../data/case1/train1.csv")
test=pd.read_csv("../data/case1/test1.csv")
for df in (train, test):
    df["n"] = (df["speed_rpm"] / 1000)
    df["Q"] = df["flow_m3_h"]
    df["P"] = df["power_w"]
    df["deltaP"] = (df["outlet_pressure_bar"] - df["inlet_pressure_bar"])

features = ["n", "P", "deltaP"]
X, y= train[features], train["Q"]
X_test, y_test= test[features], test["Q"]

X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, random_state=42)

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [231]:
def inputs(X, y, batch_size):
    batch = []
    y_values = []
    nums = rd.sample(range(len(X)), batch_size)
    for num in nums:
        feature_values = X[num]
        batch.append(feature_values)
        y_values.append(np.array(y)[num])
    return batch, y_values

def find_weights(neurons, input, output):
    weights = []
    for _ in range(neurons):
        a = np.random.randn(output, input) * np.sqrt(2 / input)
        a = a.tolist()[0]
        weights.append(a)
    return np.array(weights).T

def bias(neurons):
    return np.zeros(neurons)  

def Z(features, weights):
    return features @ weights

def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return (1/ (1+np.exp(Z)))

def linear(Z):
    return (Z).sum(axis=1)

def MSE(pred, y):
    return sum((pred - y)**2) / len(y)


In [232]:
feature_values = inputs(X_train_scaled, y_train, 5)[0]
y_true = inputs(X_train_scaled, y_train, 5)[1]
weights1 = find_weights(3, 3, 3)
weights2 = find_weights(3, 3, 1)
weights3 = find_weights(1, 3, 1)

In [233]:
Z_1 = Z(feature_values, weights1)
A_1 = relu(Z_1)
Z_2 = Z(A_1, weights2)
A_2 = sigmoid(Z_2)
Z_3 = Z(A_2, weights3)
y_pred = linear(Z_3)
error = MSE(y_pred, y_true)

In [234]:
error

np.float64(331.57958864164226)

In [262]:
feature_values = inputs(X_train_scaled, y_train, 5)[0]
y_true = inputs(X_train_scaled, y_train, 5)[1]
weights1 = find_weights(3, 3, 1)
weights2 = find_weights(1, 3, 1)

In [314]:
weights1

array([[-1.01337845, -0.31948236,  2.1821987 ],
       [-0.07283029, -1.164307  , -1.18743595],
       [-0.03637657, -0.54850574,  0.69605937]])

In [263]:
Z_1 = Z(feature_values, weights1)
A_1 = relu(Z_1)
Z_2 = Z(A_1, weights2)
output = linear(Z_2)
error = MSE(output, y_true)

In [297]:
learning_rate = 0.5
gradient_z = (2 / len(y_true)) * (np.sum(output - y_true))
gradient_weights = gradient_z * A_1

In [283]:
gradient_weight2 = np.average(gradient_weights, axis=0)

In [328]:
gradient_weight2

array([-16.08488813, -24.93270787, -15.21804866])

In [310]:
new_weight2 = weights2.flatten() - (learning_rate * gradient_weight2)

In [312]:
new_weight2

array([ 7.84574396, 12.11185989,  7.67108761])

In [ ]:
relu_grad = (A_1 > 0).astype(int)

In [335]:
(A_1 > 0).astype(int) 

array([[1, 1, 0],
       [0, 0, 1],
       [0, 1, 0],
       [1, 1, 0],
       [1, 1, 0]])

In [332]:
np.dot(relu_grad, gradient_weight2)

array([-41.017596  , -15.21804866, -24.93270787, -41.017596  ,
       -41.017596  ])

In [336]:
weights2

array([[-0.1967001 ],
       [-0.35449405],
       [ 0.06206328]])